In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("walmart_retail.sales.silver_sales")

print("Silver rows:", silver_df.count())

Silver rows: 8399


In [0]:
fact_sales = (
    silver_df
    .select(
        "row_id",
        "order_id",
        "order_date",
        "ship_date",
        "customer_name",
        "customer_segment",
        "city",
        "state",
        "zip_code",
        "region",
        "product_name",
        "product_category",
        "product_sub_category",
        "product_container",
        "order_priority",
        "ship_mode",
        "order_quantity",
        "unit_price",
        "discount",
        "sales",
        "profit",
        "shipping_cost",
        "product_base_margin",
        "number_of_records"
    )
)

print("Fact rows:", fact_sales.count())
print("Fact columns:", len(fact_sales.columns))

display(fact_sales.limit(10))

Fact rows: 8399
Fact columns: 24


row_id,order_id,order_date,ship_date,customer_name,customer_segment,city,state,zip_code,region,product_name,product_category,product_sub_category,product_container,order_priority,ship_mode,order_quantity,unit_price,discount,sales,profit,shipping_cost,product_base_margin,number_of_records
4031,28774,2012-01-01,2012-01-02,Jessica Myrick,Small Business,McKeesport,Pennsylvania,15131,East,"Perma STOR-ALL™ Hanging File Box, 13 1/8""W x 12 1/4""D x 10 1/2""H",Office Supplies,Storage & Organization,Small Box,High,Regular Air,32,5.98,0.1,180.36,-111.8,4.69,0.68,1
1914,13729,2012-01-01,2012-01-03,Matt Collister,Home Office,Bowie,Maryland,20715,East,Safco Industrial Wire Shelving,Office Supplies,Storage & Organization,Large Box,Not Specified,Express Air,9,95.99,0.08,872.48,-342.91,35.0,null,1
5272,37537,2012-01-02,2012-01-02,Alan Schoenberger,Corporate,Napa,California,94559,West,Hon 4070 Series Pagoda™ Armless Upholstered Stacking Chairs,Furniture,Chairs & Chairmats,Jumbo Drum,Low,Delivery Truck,4,291.73,0.0,1239.06,-193.08,48.8,0.56,1
6225,44069,2012-01-02,2012-01-02,Elizabeth Moffitt,Consumer,Montebello,California,90640,West,White GlueTop Scratch Pads,Office Supplies,Paper,Wrap Bag,Critical,Regular Air,43,15.04,0.08,614.8,247.79,1.97,0.39,1
5273,37537,2012-01-02,2012-01-04,Alan Schoenberger,Corporate,Napa,California,94559,West,Hon Valutask™ Swivel Chairs,Furniture,Chairs & Chairmats,Jumbo Drum,Low,Delivery Truck,43,100.98,0.07,4083.19,-1049.85,45.0,0.69,1
6224,44069,2012-01-02,2012-01-04,Elizabeth Moffitt,Consumer,Montebello,California,90640,West,"Black Print Carbonless Snap-Off® Rapid Letter, 8 1/2"" x 7""",Office Supplies,Paper,Wrap Bag,Critical,Express Air,16,9.11,0.09,137.63,26.71,2.15,0.4,1
1279,9285,2012-01-02,2012-01-04,David Philippe,Consumer,Prior Lake,Minnesota,55372,Central,"Avery Trapezoid Ring Binder, 3"" Capacity, Black, 1040 sheets",Office Supplies,Binders and Binder Accessories,Small Box,Critical,Regular Air,3,40.98,0.06,124.81,-11.937,2.99,0.36,1
5274,37537,2012-01-02,2012-01-09,Alan Schoenberger,Corporate,Napa,California,94559,West,"Dual Level, Single-Width Filing Carts",Office Supplies,Storage & Organization,Small Box,Low,Regular Air,32,155.06,0.05,4902.38,1438.49,7.07,0.59,1
5705,40354,2012-01-03,2012-01-04,Patrick Jones,Home Office,Phenix City,Alabama,36869,South,Bush Advantage Collection® Round Conference Table,Furniture,Tables,Jumbo Box,High,Delivery Truck,4,212.6,0.09,698.0,-93.16,52.2,0.64,1
1336,9762,2012-01-03,2012-01-04,Larry Tron,Home Office,Draper,Utah,84020,West,36X48 HARDFLOOR CHAIRMAT,Furniture,Office Furnishings,Medium Box,High,Regular Air,12,20.98,0.05,262.76,-146.05,21.2,0.78,1


In [0]:
fact_sales = (
    fact_sales
    .withColumn(
        "gross_sales",
        F.round(F.col("unit_price") * F.col("order_quantity"), 2)
    )
    .withColumn(
        "discount_amount",
        F.round(
            F.col("unit_price") * F.col("order_quantity") * F.col("discount"),
            2
        )
    )
    .withColumn(
        "profit_margin_pct",
        F.when(
            F.col("sales") != 0,
            F.round((F.col("profit") / F.col("sales")) * 100, 2)
        )
    )
    .withColumn(
        "shipping_days",
        F.datediff(F.col("ship_date"), F.col("order_date"))
    )
)

display(
    fact_sales.select(
        "row_id",
        "order_id",
        "order_date",
        "sales",
        "profit",
        "gross_sales",
        "discount_amount",
        "profit_margin_pct",
        "shipping_days"
    ).limit(10)
)

row_id,order_id,order_date,sales,profit,gross_sales,discount_amount,profit_margin_pct,shipping_days
4031,28774,2012-01-01,180.36,-111.8,191.36,19.14,-61.99,1
1914,13729,2012-01-01,872.48,-342.91,863.91,69.11,-39.3,2
5272,37537,2012-01-02,1239.06,-193.08,1166.92,0.0,-15.58,0
6225,44069,2012-01-02,614.8,247.79,646.72,51.74,40.3,0
5273,37537,2012-01-02,4083.19,-1049.85,4342.14,303.95,-25.71,2
6224,44069,2012-01-02,137.63,26.71,145.76,13.12,19.41,2
1279,9285,2012-01-02,124.81,-11.937,122.94,7.38,-9.56,2
5274,37537,2012-01-02,4902.38,1438.49,4961.92,248.1,29.34,7
5705,40354,2012-01-03,698.0,-93.16,850.4,76.54,-13.35,1
1336,9762,2012-01-03,262.76,-146.05,251.76,12.59,-55.58,1


In [0]:
dim_customer = (
    silver_df
    .select(
        "customer_name",
        "customer_segment"
    )
    .dropDuplicates()
    .withColumn(
        "customer_key",
        F.monotonically_increasing_id()
    )
)

print("Customer dimension rows:", dim_customer.count())

display(dim_customer.limit(10))

Customer dimension rows: 989


customer_name,customer_segment,customer_key
Jessica Myrick,Small Business,0
Matt Collister,Home Office,1
Alan Schoenberger,Corporate,2
Elizabeth Moffitt,Consumer,3
David Philippe,Consumer,4
Patrick Jones,Home Office,5
Larry Tron,Home Office,6
Alex Russell,Home Office,7
Bill Donatelli,Corporate,8
Ann Steele,Corporate,9


In [0]:
dim_product = (
    silver_df
    .select(
        "product_name",
        "product_category",
        "product_sub_category",
        "product_container",
        "product_base_margin"
    )
    .dropDuplicates()
    .withColumn(
        "product_key",
        F.monotonically_increasing_id()
    )
)

print("Product dimension rows:", dim_product.count())

display(dim_product.limit(10))

Product dimension rows: 1289


product_name,product_category,product_sub_category,product_container,product_base_margin,product_key
"Perma STOR-ALL™ Hanging File Box, 13 1/8""W x 12 1/4""D x 10 1/2""H",Office Supplies,Storage & Organization,Small Box,0.68,0
Safco Industrial Wire Shelving,Office Supplies,Storage & Organization,Large Box,null,1
Hon 4070 Series Pagoda™ Armless Upholstered Stacking Chairs,Furniture,Chairs & Chairmats,Jumbo Drum,0.56,2
White GlueTop Scratch Pads,Office Supplies,Paper,Wrap Bag,0.39,3
Hon Valutask™ Swivel Chairs,Furniture,Chairs & Chairmats,Jumbo Drum,0.69,4
"Black Print Carbonless Snap-Off® Rapid Letter, 8 1/2"" x 7""",Office Supplies,Paper,Wrap Bag,0.4,5
"Avery Trapezoid Ring Binder, 3"" Capacity, Black, 1040 sheets",Office Supplies,Binders and Binder Accessories,Small Box,0.36,6
"Dual Level, Single-Width Filing Carts",Office Supplies,Storage & Organization,Small Box,0.59,7
Bush Advantage Collection® Round Conference Table,Furniture,Tables,Jumbo Box,0.64,8
36X48 HARDFLOOR CHAIRMAT,Furniture,Office Furnishings,Medium Box,0.78,9


In [0]:
dim_date = (
    silver_df
    .select("order_date")
    .dropDuplicates()
    .withColumn(
        "date_key",
        F.date_format("order_date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("order_date"))
    .withColumn("quarter", F.quarter("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("month_name", F.date_format("order_date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("order_date"))
    .withColumn("day_of_month", F.dayofmonth("order_date"))
    .withColumn("day_of_week", F.dayofweek("order_date"))
    .withColumn("day_name", F.date_format("order_date", "EEEE"))
)

print("Date dimension rows:", dim_date.count())

display(
    dim_date.orderBy("order_date").limit(10)
)

Date dimension rows: 1418


order_date,date_key,year,quarter,month,month_name,week_of_year,day_of_month,day_of_week,day_name
2012-01-01,20120101,2012,1,1,January,52,1,1,Sunday
2012-01-02,20120102,2012,1,1,January,1,2,2,Monday
2012-01-03,20120103,2012,1,1,January,1,3,3,Tuesday
2012-01-04,20120104,2012,1,1,January,1,4,4,Wednesday
2012-01-05,20120105,2012,1,1,January,1,5,5,Thursday
2012-01-06,20120106,2012,1,1,January,1,6,6,Friday
2012-01-07,20120107,2012,1,1,January,1,7,7,Saturday
2012-01-08,20120108,2012,1,1,January,1,8,1,Sunday
2012-01-09,20120109,2012,1,1,January,2,9,2,Monday
2012-01-10,20120110,2012,1,1,January,2,10,3,Tuesday


In [0]:
dim_location = (
    silver_df
    .select(
        "city",
        "state",
        "zip_code",
        "region"
    )
    .dropDuplicates()
    .withColumn(
        "location_key",
        F.monotonically_increasing_id()
    )
)

print("Location dimension rows:", dim_location.count())

display(dim_location.limit(10))

Location dimension rows: 1636


city,state,zip_code,region,location_key
McKeesport,Pennsylvania,15131,East,0
Bowie,Maryland,20715,East,1
Napa,California,94559,West,2
Montebello,California,90640,West,3
Prior Lake,Minnesota,55372,Central,4
Phenix City,Alabama,36869,South,5
Draper,Utah,84020,West,6
Lake Oswego,Oregon,97035,West,7
Kearney,Nebraska,68847,Central,8
Layton,Utah,84041,West,9


In [0]:
fact_sales_gold = (
    fact_sales.alias("f")

    # Customer
    .join(
        dim_customer.alias("c"),
        (F.col("f.customer_name") == F.col("c.customer_name")) &
        (F.col("f.customer_segment") == F.col("c.customer_segment")),
        "left"
    )

    # Product
    .join(
        dim_product.alias("p"),
        (F.col("f.product_name") == F.col("p.product_name")) &
        (F.col("f.product_category") == F.col("p.product_category")) &
        (F.col("f.product_sub_category") == F.col("p.product_sub_category")) &
        (F.col("f.product_container") == F.col("p.product_container")),
        "left"
    )

    # Date
    .join(
        dim_date.alias("d"),
        F.col("f.order_date") == F.col("d.order_date"),
        "left"
    )

    # Location
    .join(
        dim_location.alias("l"),
        (F.col("f.city") == F.col("l.city")) &
        (F.col("f.state") == F.col("l.state")) &
        (F.col("f.zip_code") == F.col("l.zip_code")) &
        (F.col("f.region") == F.col("l.region")),
        "left"
    )

    .select(
        "f.*",
        "c.customer_key",
        "p.product_key",
        "d.date_key",
        "l.location_key"
    )
)

print("Gold fact rows:", fact_sales_gold.count())
print("Gold fact columns:", len(fact_sales_gold.columns))

display(
    fact_sales_gold.select(
        "row_id",
        "order_id",
        "customer_key",
        "product_key",
        "date_key",
        "location_key",
        "sales",
        "profit",
        "profit_margin_pct"
    ).limit(10)
)

Gold fact rows: 8697
Gold fact columns: 32


row_id,order_id,customer_key,product_key,date_key,location_key,sales,profit,profit_margin_pct
4031,28774,0,708,20120101,381,180.36,-111.8,-61.99
1914,13729,192,543,20120101,1538,872.48,-342.91,-39.3
5272,37537,797,208,20120102,690,1239.06,-193.08,-15.58
6225,44069,245,794,20120102,382,614.8,247.79,40.3
5273,37537,797,375,20120102,690,4083.19,-1049.85,-25.71
6224,44069,245,0,20120102,382,137.63,26.71,19.41
1279,9285,496,461,20120102,597,124.81,-11.937,-9.56
5274,37537,797,544,20120102,690,4902.38,1438.49,29.34
5705,40354,193,827,20120103,0,698.0,-93.16,-13.35
1336,9762,856,884,20120103,691,262.76,-146.05,-55.58


In [0]:
print("Customer duplicate keys:")
display(
    dim_customer
    .groupBy("customer_name", "customer_segment")
    .count()
    .filter(F.col("count") > 1)
)

print("Product duplicate keys:")
display(
    dim_product
    .groupBy(
        "product_name",
        "product_category",
        "product_sub_category",
        "product_container"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Date duplicate keys:")
display(
    dim_date
    .groupBy("order_date")
    .count()
    .filter(F.col("count") > 1)
)

print("Location duplicate keys:")
display(
    dim_location
    .groupBy("city", "state", "zip_code", "region")
    .count()
    .filter(F.col("count") > 1)
)

Customer duplicate keys:


customer_name,customer_segment,count


Product duplicate keys:


product_name,product_category,product_sub_category,product_container,count
Bush Advantage Collection® Round Conference Table,Furniture,Tables,Jumbo Box,2
"Fellowes Mobile Numeric Keypad, Graphite",Technology,Computer Peripherals,Small Box,2
"Iceberg OfficeWorks 42"" Round Tables",Furniture,Tables,Jumbo Box,2
Keytronic French Keyboard,Technology,Computer Peripherals,Small Box,2
Panasonic KX-P3200 Dot Matrix Printer,Technology,Office Machines,Jumbo Drum,2
"Bevis Round Conference Table Top, X-Base",Furniture,Tables,Jumbo Box,2
Keytronic 105-Key Spanish Keyboard,Technology,Computer Peripherals,Small Box,2
BoxOffice By Design Rectangular and Half-Moon Meeting Room Tables,Furniture,Tables,Jumbo Box,2
"Micro Innovations Micro Digital Wireless Keyboard and Mouse, Gray",Technology,Computer Peripherals,Small Box,2
"Bevis Round Bullnose 29"" High Table Top",Furniture,Tables,Jumbo Box,2


Date duplicate keys:


order_date,count


Location duplicate keys:


city,state,zip_code,region,count


In [0]:
dim_product = (
    silver_df
    .groupBy(
        "product_name",
        "product_category",
        "product_sub_category",
        "product_container"
    )
    .agg(
        F.first(
            "product_base_margin",
            ignorenulls=True
        ).alias("product_base_margin")
    )
    .withColumn(
        "product_key",
        F.monotonically_increasing_id()
    )
)

print("Product dimension rows:", dim_product.count())

display(dim_product.limit(10))

Product dimension rows: 1264


product_name,product_category,product_sub_category,product_container,product_base_margin,product_key
"Perma STOR-ALL™ Hanging File Box, 13 1/8""W x 12 1/4""D x 10 1/2""H",Office Supplies,Storage & Organization,Small Box,0.68,0
Safco Industrial Wire Shelving,Office Supplies,Storage & Organization,Large Box,null,1
Hon 4070 Series Pagoda™ Armless Upholstered Stacking Chairs,Furniture,Chairs & Chairmats,Jumbo Drum,0.56,2
White GlueTop Scratch Pads,Office Supplies,Paper,Wrap Bag,0.39,3
Hon Valutask™ Swivel Chairs,Furniture,Chairs & Chairmats,Jumbo Drum,0.69,4
"Black Print Carbonless Snap-Off® Rapid Letter, 8 1/2"" x 7""",Office Supplies,Paper,Wrap Bag,0.4,5
"Avery Trapezoid Ring Binder, 3"" Capacity, Black, 1040 sheets",Office Supplies,Binders and Binder Accessories,Small Box,0.36,6
"Dual Level, Single-Width Filing Carts",Office Supplies,Storage & Organization,Small Box,0.59,7
Bush Advantage Collection® Round Conference Table,Furniture,Tables,Jumbo Box,0.64,8
36X48 HARDFLOOR CHAIRMAT,Furniture,Office Furnishings,Medium Box,0.78,9


In [0]:
print("Duplicate product keys:")

display(
    dim_product
    .groupBy(
        "product_name",
        "product_category",
        "product_sub_category",
        "product_container"
    )
    .count()
    .filter(F.col("count") > 1)
)

Duplicate product keys:


product_name,product_category,product_sub_category,product_container,count


In [0]:
fact_sales_gold = (
    fact_sales.alias("f")
    .join(
        dim_customer.alias("c"),
        (F.col("f.customer_name") == F.col("c.customer_name")) &
        (F.col("f.customer_segment") == F.col("c.customer_segment")),
        "left"
    )
    .join(
        dim_product.alias("p"),
        (F.col("f.product_name") == F.col("p.product_name")) &
        (F.col("f.product_category") == F.col("p.product_category")) &
        (F.col("f.product_sub_category") == F.col("p.product_sub_category")) &
        (F.col("f.product_container") == F.col("p.product_container")),
        "left"
    )
    .join(
        dim_date.alias("d"),
        F.col("f.order_date") == F.col("d.order_date"),
        "left"
    )
    .join(
        dim_location.alias("l"),
        (F.col("f.city") == F.col("l.city")) &
        (F.col("f.state") == F.col("l.state")) &
        (F.col("f.zip_code") == F.col("l.zip_code")) &
        (F.col("f.region") == F.col("l.region")),
        "left"
    )
    .select(
        "f.*",
        "c.customer_key",
        "p.product_key",
        "d.date_key",
        "l.location_key"
    )
)

print("Gold fact rows:", fact_sales_gold.count())
print("Gold fact columns:", len(fact_sales_gold.columns))

Gold fact rows: 8399
Gold fact columns: 32


In [0]:
print("Null surrogate keys:")

display(
    fact_sales_gold.select(
        F.count(
            F.when(F.col("customer_key").isNull(), 1)
        ).alias("null_customer_keys"),

        F.count(
            F.when(F.col("product_key").isNull(), 1)
        ).alias("null_product_keys"),

        F.count(
            F.when(F.col("date_key").isNull(), 1)
        ).alias("null_date_keys"),

        F.count(
            F.when(F.col("location_key").isNull(), 1)
        ).alias("null_location_keys")
    )
)

Null surrogate keys:


null_customer_keys,null_product_keys,null_date_keys,null_location_keys
0,0,0,0


In [0]:
gold_fact_table = "walmart_retail.sales.gold_fact_sales"

(
    fact_sales_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_fact_table)
)

print(f"Saved: {gold_fact_table}")

Saved: walmart_retail.sales.gold_fact_sales


In [0]:
gold_customer_table = "walmart_retail.sales.gold_dim_customer"

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_customer_table)
)

print(f"Saved: {gold_customer_table}")

Saved: walmart_retail.sales.gold_dim_customer


In [0]:
gold_product_table = "walmart_retail.sales.gold_dim_product"

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_product_table)
)

print(f"Saved: {gold_product_table}")

Saved: walmart_retail.sales.gold_dim_product


In [0]:
gold_date_table = "walmart_retail.sales.gold_dim_date"

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_date_table)
)

print(f"Saved: {gold_date_table}")

Saved: walmart_retail.sales.gold_dim_date


In [0]:
gold_location_table = "walmart_retail.sales.gold_dim_location"

(
    dim_location.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_location_table)
)

print(f"Saved: {gold_location_table}")

Saved: walmart_retail.sales.gold_dim_location


In [0]:
tables = [
    "walmart_retail.sales.bronze_sales",
    "walmart_retail.sales.silver_sales",
    "walmart_retail.sales.gold_fact_sales",
    "walmart_retail.sales.gold_dim_customer",
    "walmart_retail.sales.gold_dim_product",
    "walmart_retail.sales.gold_dim_date",
    "walmart_retail.sales.gold_dim_location"
]

for table in tables:
    print(f"{table}: {spark.table(table).count():,} rows")

walmart_retail.sales.bronze_sales: 8,399 rows
walmart_retail.sales.silver_sales: 8,399 rows
walmart_retail.sales.gold_fact_sales: 8,399 rows
walmart_retail.sales.gold_dim_customer: 989 rows
walmart_retail.sales.gold_dim_product: 1,264 rows
walmart_retail.sales.gold_dim_date: 1,418 rows
walmart_retail.sales.gold_dim_location: 1,636 rows
